In [ ]:
import pandas as pd
import string
import random
import time
import optuna
import openai
from sklearn.metrics import f1_score

# Configure API key and paths
openai.api_key = ""
excel_path = "/path/to/prompt_parts.xlsx"
output_path = "/path/to/results.xlsx"

# Load prompt parts
roles_df = pd.read_excel(excel_path, sheet_name="roles")
tasks_df = pd.read_excel(excel_path, sheet_name="tasks")
instructions_df = pd.read_excel(excel_path, sheet_name="general_instructions")
questions_df = pd.read_excel(excel_path, sheet_name="user_questions")
roles = roles_df['role'].dropna().tolist()
tasks = tasks_df['task'].dropna().tolist()
general_instructions = instructions_df['instruction'].dropna().tolist()
user_questions = questions_df['question'].dropna().tolist()

# GPT-3.5 query function
def ask_question(prompt, question, temperature=0.9):
    for retry in range(5):
        try:
            response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "system", "content": prompt}, {"role": "user", "content": question}],
                max_tokens=800,
                temperature=temperature,
            )
            raw = response.choices[0].message.content.strip().lower()
            cleaned = raw.translate(str.maketrans('', '', string.punctuation)).strip().lower()
            return raw, cleaned
        except openai.APIError as e:
            if retry == 4:
                return None, None
            time.sleep(2 ** retry)
        except Exception:
            return None, None

def summarize_relation(value):
    prompt = f"Summarize term '{value}' to one of: activation, inhibition, phosphorylation, or no information."
    _, cleaned = ask_question("", prompt, temperature=0)
    return cleaned

# Compute F1 metrics
def calculate_metrics(df):
    relations = ['activation', 'inhibition', 'phosphorylation']
    df['pred_processed'] = df['predict_relation'].apply(
        lambda x: x if x in relations or x == 'no information' else summarize_relation(x)
    )
    scores = {}
    for rel in relations:
        y_true = df['relation'] == rel
        y_pred = df['pred_processed'] == rel
        scores[rel] = f1_score(y_true, y_pred, zero_division=0)
    scores['micro'] = f1_score(df['relation'], df['pred_processed'], average='micro', zero_division=0)
    return scores

# Evaluate prompt fitness
def get_fitness(prompt, question_pattern):
    df = pd.read_csv("/content/drive/MyDrive/LLM model/training.csv")
    dp = (
        "Example: Q: What effect does gene EGF have on gene EGFR? A: Activation. "
        "Q: What effect does gene GRK2 have on gene OR2AJ1? A: Inhibition. "
        "Q: What effect does gene CDK9 have on gene NELFB? A: Phosphorylation. "
        "Answers must be one of activation, inhibition, phosphorylation or no information."
    )
    records = []
    for _, row in df.sample(frac=1).iterrows():
        g1, g2 = row['starter'].upper(), row['receiver'].upper()
        true_rel = row['relation_name']
        question = question_pattern.format(gene1=g1, gene2=g2)
        raw, pred = ask_question(f"{prompt}\n{dp}", question)
        records.append({'relation': true_rel, 'predict_relation': pred or 'no information'})
    metrics = calculate_metrics(pd.DataFrame(records))
    return metrics['micro']

def assemble_prompt(role, task, instruction, question):
    return f"{role}, {task}, {instruction}, {question}"

# Optuna objective
def objective(trial):
    role = trial.suggest_categorical('role', roles)
    task = trial.suggest_categorical('task', tasks)
    instruction = trial.suggest_categorical('instruction', general_instructions)
    question = trial.suggest_categorical('question', user_questions)
    prompt = assemble_prompt(role, task, instruction, question)
    ascii_key = ''.join(random.choices(string.ascii_letters + string.digits + string.punctuation, k=15))
    trial.set_user_attr('ascii_key', ascii_key)
    return get_fitness(prompt, question)

# Run optimization
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study.optimize(objective, n_trials=5)

# Collect results
trials_data = []
for t in study.trials:
    trials_data.append({
        'trial_number': t.number,
        'f1_score': t.value,
        'ascii_key': t.user_attrs.get('ascii_key'),
        'role': t.params['role'],
        'task': t.params['task'],
        'instruction': t.params['instruction'],
        'question': t.params['question'],
    })
all_trials_df = pd.DataFrame(trials_data)

best = study.best_trial
best_data = {
    'trial_number': best.number,
    'f1_score': best.value,
    'ascii_key': best.user_attrs.get('ascii_key'),
    'role': best.params['role'],
    'task': best.params['task'],
    'instruction': best.params['instruction'],
    'question': best.params['question'],
}
best_trial_df = pd.DataFrame([best_data])

# Save to Excel
with pd.ExcelWriter(output_path) as writer:
    all_trials_df.to_excel(writer, sheet_name='All Trials', index=False)
    best_trial_df.to_excel(writer, sheet_name='Best Trial', index=False)

print(f"Results saved to {output_path}")
